# Valutazione della qualità della Knowledge Base

In questo esercizio esplora come confrontare frasi e documenti in modo automatico mediante embedding semantici. L’obiettivo è verificare se i chunk di testo contengono informazioni ridondanti, discordanti o mancanti rispetto a un insieme di riferimenti.

Partendo da un piccolo database di frasi in italiano, costruzione delle rappresentazioni vettoriali con un modello multilingual SentenceTransformer, calcolo della similarità tra embedding e valutazione:

- ridondanza semantica: coppie di chunk molto simili tra loro;
- copertura semantica: quanto bene i chunk coprono richieste di riferimento;
- discordanza logica: casi in cui la similarità semantica appare elevata ma la semantica reale differisce.

L'esercizio combina: embedding semantici, confronto tra similarità numerica e regole ASP, visualizzazione PCA 3D dei punti nello spazio semantico.

L'obiettivo finale è interpretare i risultati e verificare come il modello distingue tra informazioni coerenti, sovrapposte e inconsistenti.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

In [ ]:
DATABASE = {
    # Topic Auto
    "chunk_1": "Mario possiede una Citroen nera.",
    "chunk_2": "Dopo anni di onorato servizio, Mario è riuscito a comprare una Citroen nera.",
    "chunk_3": "Una bellissima Citroen di colore nero è stata acquistata dal signor Mario due settimane fa.",
    "chunk_4": "Mario, lavoratore scrupoloso, decise di comprare una Citroen verde.",
    # Topic Città
    "chunk_5": "Una delle città più popolose dell'Inghilterra è Manchester.",
    "chunk_6": "La capitale della Francia è Parigi.",
    "chunk_7": "La Francia è uno stato con capitale Bordeaux.",
    # Topic Orari di servizio
    "chunk_8": "Il negozio è aperto dal lunedì al venerdì, dalle 9 alle 18.",
    "chunk_9": "Dal lunedì al venerdì l'orario di apertura del negozio va dalle 9:00 alle 18:00.",
    "chunk_10": "Il sabato il negozio è aperto dalle 10 alle 13.",
    "chunk_11": "Il negozio, il sabato, resta aperto dalle 10:00 fino alle 16:00.",
    "chunk_12": "",  # buco voluto: nessuna frase parla della domenica -> se hai una query di riferimento "orario domenica", risulterà scoperta
}

EMBED = "paraphrase-multilingual-MiniLM-L12-v2"

TOPIC = {
    "chunk_1": "auto", "chunk_2": "auto", "chunk_3": "auto", "chunk_4": "auto",
    "chunk_5": "citta", "chunk_6": "citta", "chunk_7": "citta",
    "chunk_8": "orario", "chunk_9": "orario", "chunk_10": "orario", "chunk_11": "orario"
}

COLORS = {
    "auto": "tab:blue", "citta": "tab:orange", "orario": "tab:green"
}

REFER = {
    "capitale_francia": "Qual è la capitale della Francia?",
    "capitale_italia": "Qual è la capitale dell'Italia?",
    "orario_domenica": "Qual è l'orario di apertura del negozio la domenica?"
}

ASP_RESULT = {
    "ridondanza": [
        ("chunk_1", "chunk_2"), ("chunk_1", "chunk_3"), ("chunk_2", "chunk_3"),  # colore nera
        ("chunk_1", "chunk_2"), ("chunk_1", "chunk_3"), ("chunk_1", "chunk_4"),
        ("chunk_2", "chunk_3"), ("chunk_2", "chunk_4"), ("chunk_3", "chunk_4"),  # auto mario/citroen
        ("chunk_8", "chunk_9"),  # orario lun_ven
    ],
    "discordanza": [
        ("chunk_6", "chunk_7"),   # capitale
        ("chunk_1", "chunk_4"), ("chunk_2", "chunk_4"), ("chunk_3", "chunk_4"),  # colore nera/verde
        ("chunk_10", "chunk_11"),  # orario sabato
    ],
    "non_coperto": ["capitale_italia", "orario_domenica"],
}

## Spiegazione delle funzioni

- load_model(): Carica il modello multilingual SentenceTransformer scelto (in questo caso 'paraphrase-multilingual-MiniLM-L12-v2'). Il modello trasforma frasi e documenti in vettori semantici, così da poterli confrontare in modo quantitativo.
- set_embedding(model, chunks): Filtra i chunk non vuoti, esclude eventuali testi vuoti o composti solo da spazi e li codifica con il modello. Ogni frase viene trasformata in un embedding normalizzato, cioè in un vettore numerico pronto per il confronto. La funzione restituisce anche la lista degli identificatori dei chunk validi, mantenendo la corrispondenza tra ciascun testo e il relativo vettore.
- matrix_sim(emb): A partire dalla matrice degli embedding, calcola la matrice di similarità semantica tramite prodotto scalare. In pratica, ogni cella (i, j) indica quanto il chunk i sia simile al chunk j. Questa matrice è il punto di partenza per individuare ridondanza, copertura e discordanza tra i testi.

In [ ]:
def load_model() -> SentenceTransformer:
    return SentenceTransformer(EMBED)

def set_embedding(model: SentenceTransformer, chunks: dict[str, str]) -> tuple[list[str], np.ndarray]:
    ids = [k for k, v in chunks.items() if v.strip()]
    if not ids:
        return [], np.empty((0, 0))
    emb = model.encode([chunks[k] for k in ids], normalize_embeddings=True)
    return ids, np.asarray(emb)


def matrix_sim(emb: np.ndarray) -> np.ndarray:
    if emb.size == 0:
        return np.empty((0, 0))
    return emb @ emb.T


## Main functions

- redundancy(ids, sim, thresholds): Individuare coppie di chunk semanticamente molto simili tra loro, cioè potenzialmente ridondanti. Calcola le coppie presenti nel triangolo superiore della matrice con np.triu_indices(..., k=1) (questo evita di considerare: l'auto-confronto (chunk con se stesso) e coppie duplicate in forma simmetrica (i, j) e (j, i)). Per ogni soglia, seleziona solo le coppie con similarità >= thresh; salva il risultato in un dizionario del tipo: {soglia: [(id_chunk_a, id_chunk_b, similarita), ...]}.
- coverage(model, chunks_ids, emb_chunks, riferimenti, thresh=0.5): Verificare se i chunk presenti nel database coprono adeguatamente le domande di riferimento. Codifica le domande di riferimento con model.encode(..., normalize_embeddings=True). Calcola la matrice di similarità tra chunk e riferimenti: emb_chunks @ emb_rif.T, per ogni riferimento: trova il chunk con similarità massima; confronta quel valore con la soglia impostata; memorizza un risultato del tipo: (coperto, miglior_chunk, similarita_massima). Il valore booleano indica se la domanda è stata coperta o meno.

In [ ]:
def redundancy(ids: list[str], sim: np.ndarray, thresholds: list[float]) -> dict[float, list[tuple[str, str, float]]]:
    n = len(ids)
    if n < 2 or sim.size == 0:
        return {t: [] for t in thresholds}

    # Estrae gli indici del triangolo superiore (evita auto-confronti e duplicati simmetrici)
    upper_i, upper_j = np.triu_indices(n, k=1)
    pair_sims = sim[upper_i, upper_j]

    results: dict[float, list[tuple[str, str, float]]] = {}
    for thresh in thresholds:
        mask = pair_sims >= thresh
        selected_pairs = [
            (ids[i], ids[j], round(float(s), 3))
            for i, j, s in zip(upper_i[mask], upper_j[mask], pair_sims[mask])
        ]
        results[thresh] = selected_pairs

    return results


def coverage(model: SentenceTransformer, chunks_ids: list[str], emb_chunks: np.ndarray, riferimenti: dict[str, str], thresh: float = 0.5) -> dict[str, tuple[bool, str, float]]:
    if not riferimenti or len(chunks_ids) == 0:
        return {}

    emb_rif = model.encode(list(riferimenti.values()), normalize_embeddings=True)
    sim_matrix = emb_chunks @ emb_rif.T  # Forma: (n_chunks, n_riferimenti)

    results: dict[str, tuple[bool, str, float]] = {}
    for idx_rif, key in enumerate(riferimenti.keys()):
        sims_col = sim_matrix[:, idx_rif]
        idx_best = int(np.argmax(sims_col))
        best_sim = float(sims_col[idx_best])
        results[key] = (best_sim >= thresh, chunks_ids[idx_best], round(best_sim, 3))

    return results

In [ ]:
def plot_3d(ids: list[str], emb: np.ndarray, topic: dict[str, str]) -> None:
    if len(ids) < 3:
        print("Impossibile generare PCA 3D: necessari almeno 3 chunk.")
        return

    n_comp = min(3, emb.shape[0], emb.shape[1])
    pca = PCA(n_components=n_comp)
    coords = pca.fit_transform(emb)

    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")

    for cid, (x, y, z) in zip(ids, coords):
        colore = COLORS.get(topic.get(cid, ""), "gray")
        ax.scatter(x, y, z, color=colore, s=60, edgecolors="k", alpha=0.85)
        # Piccolo offset visivo per non sovrapporre il testo al punto
        ax.text(x + 0.01, y + 0.01, z + 0.01, cid.replace("chunk_", "c"), fontsize=8)

    # Proxy artists per la legenda dei topic
    for nome_topic, colore in COLORS.items():
        ax.scatter([], [], [], color=colore, label=nome_topic)
    ax.legend(loc="upper left")

    var_explained = pca.explained_variance_ratio_.sum()
    ax.set_title(f"PCA 3D chunk (Varianza spiegata: {var_explained:.1%})")
    ax.set_xlabel("Componente 1")
    ax.set_ylabel("Componente 2")
    ax.set_zlabel("Componente 3")

    plt.tight_layout()
    plt.show()

def outputs(ridondanza_emb: dict[float, list[tuple[str, str, float]]], copertura_emb: dict) -> None:
    print("-" * 60)
    print("Confronto semantico (embedding) vs logico (asp)")
    print("-" * 60)

    # Set normalizzato per lookup immediato O(1)
    discordanze_asp = set()
    for item in ASP_RESULT.get("discordanza", []):
        discordanze_asp.add((item[0], item[1]))
        discordanze_asp.add((item[1], item[0]))

    print("\n--- Ridondanza Semantica (coppie sopra soglia) ---")
    for soglia, coppie in ridondanza_emb.items():
        print(f"\nSoglia {soglia:.2f}: {len(coppie)} coppie trovate")
        for a, b, s in coppie:
            etichetta = ""
            if (a, b) in discordanze_asp:
                etichetta = "  <-- ATTENZIONE: per ASP è una DISCORDANZA logica, non ridondanza!"
            print(f"  {a} - {b} (sim={s:.3f}){etichetta}")

    print("\n--- Copertura Semantica (similarità chunk / riferimento) ---")
    for chiave, (coperto, best_chunk, sim) in copertura_emb.items():
        stato = "COPERTO" if coperto else "NON COPERTO"
        print(f"  {chiave:<15}: {stato:<12} (miglior match: {best_chunk}, sim={sim:.3f})")2

In [ ]:
model = load_model()
ids, emb = set_embedding(model, DATABASE)

if not ids:
    print("Database vuoto o nessun chunk valido trovato.")
    return

sim = matrix_sim(emb)
ridondanza = redundancy(ids, sim, thresholds=[0.75, 0.80, 0.85, 0.90, 0.95])
copertura = coverage(model, ids, emb, REFER, thresh=0.5)

outputs(ridondanza, copertura)
plot_3d(ids, emb, TOPIC)
